In [1]:
!git clone https://github.com/676627/deep-learning-eurosat.git
%cd deep-learning-eurosat
!pip install -q rasterio wandb

Cloning into 'deep-learning-eurosat'...
remote: Enumerating objects: 346, done.
remote: Counting objects: 100% (346/346), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 346 (delta 162), reused 303 (delta 122), pack-reused 0 (from 0)
Receiving objects: 100% (346/346), 7.39 MiB | 12.74 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/deep-learning-eurosat


In [2]:
# Download and unzip data
!wget -q "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"
!wget -q "https://zenodo.org/records/7711810/files/EuroSAT_MS.zip"

!unzip -q EuroSAT_RGB.zip -d data/
!unzip -q EuroSAT_MS.zip -d data/

In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 676627 (180240-h-gskulen-p-vestlandet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
!python src/train.py --mode ms --model_version batchnorm_3conv --lr 1e-4 --dropout 0.4

2026-04-26 12:42:58.673663: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777207378.695211   13789 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777207378.701676   13789 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777207378.717014   13789 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777207378.717077   13789 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777207378.717088   13789 computation_placer.cc:177] computation placer alr

In [6]:
!python src/evaluate.py

2026-04-26 14:27:04.313929: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777213624.338694   41561 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777213624.347415   41561 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777213624.368334   41561 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777213624.368396   41561 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777213624.368401   41561 computation_placer.cc:177] computation placer alr

In [7]:
import sys
sys.path.insert(0, '.')
import json
import numpy as np
from src.dataset import load_ms_dataset

# Recompute stats using the same seed as training
_, _, stats = load_ms_dataset("data/EuroSAT_MS", seed=42)

# Save to disk
with open("checkpoints/MS_batchnorm_3conv_full_stats.json", "w") as f:
    json.dump(stats, f)

print(stats)

Found 27000 images | 21600 train, 5400 val | 13 bands
Estimating normalisation stats from 2000 training images...


Sampling: 100%|██████████| 2000/2000 [00:05<00:00, 341.25it/s]


{'mean': [0.1346815973520279, 0.11094258725643158, 0.10332129895687103, 0.09370754659175873, 0.11865845322608948, 0.19760534167289734, 0.23419508337974548, 0.22691279649734497, 0.0722285583615303, 0.0012071648379787803, 0.1805816888809204, 0.11079370230436325, 0.2563565671443939], 'std': [0.02386559545993805, 0.032237451523542404, 0.0383630096912384, 0.058011263608932495, 0.0559089258313179, 0.08596274256706238, 0.1085418164730072, 0.11158045381307602, 0.040207840502262115, 0.00046486029168590903, 0.10086487978696823, 0.07580359280109406, 0.12326868623495102]}
